# Export control samples to h5ad

### Load packages

In [ ]:
#| eval: true
#| message: false 
#| warning: false

# ────────────────────────────────────────────────────────────────
# Python environment setup via reticulate
# ────────────────────────────────────────────────────────────────
library(reticulate)
use_python("/home/adaml9/scratch/miniforge3/envs/scFates/bin/python", required = TRUE)

# ────────────────────────────────────────────────────────────────
# Load R packages
# ────────────────────────────────────────────────────────────────
library(future)
library(Seurat)
library(Signac)
library(SCpubr)
library(scales)

# Annotation & reference genome packages
library(EnsDb.Hsapiens.v86)
library(BSgenome.Hsapiens.UCSC.hg38)

# ────────────────────────────────────────────────────────────────
# Source project/global R scripts
# ────────────────────────────────────────────────────────────────
source("/projects/site/pred/ihb-g-deco/USERS/adaml9/intestine_fate/notebooks/global.R")
source("/pstore/data/ihb-g-deco/USERS/adaml9/R-commons/scripts/plotting/colors.R")
source("/pstore/data/ihb-g-deco/USERS/adaml9/R-commons/scripts/plotting/fonts.R")
source("/pstore/data/ihb-g-deco/USERS/adaml9/R-commons/scripts/export/seurat_to_anndata.R")

# ────────────────────────────────────────────────────────────────
# Global defaults and options
# ────────────────────────────────────────────────────────────────
set.seed(1)

# Memory and compatibility options
options(
  future.globals.maxSize = 3e+09,
  Seurat.object.assay.version = "v3"
)


### Load data

In [ ]:
data.seurat.multiome <- readRDS(paste0(io$outdir.processed, "/parse.multiome.annotated.rds"))

In [ ]:
data.seurat.multiome.eecs <- readRDS(paste0(io$outdir.processed, "/parse.multiome.eecs.annotated.rds"))

In [ ]:
cell_data <- data.seurat.multiome@meta.data
cell_data[rownames(data.seurat.multiome.eecs@meta.data),] <- data.seurat.multiome.eecs@meta.data
data.seurat.multiome@meta.data <- cell_data

In [ ]:
data.seurat.multiome.list <- SplitObject(data.seurat.multiome, split.by="batch")

In [ ]:
data.seurat.multiome1 <- data.seurat.multiome.list$Multiome1
data.seurat.multiome1@meta.data <- data.seurat.multiome1@meta.data %>%
    as.data.frame() %>%
    rownames_to_column(var="barcode") %>%
    mutate(barcode = paste0(map_chr(barcode, ~ strsplit(.x, "-")[[1]][1]), "-ITBOGE013_YBP2_14_Multiome_1")) %>%
    column_to_rownames(var="barcode")

In [ ]:
data.seurat.multiome2 <- data.seurat.multiome.list$Multiome2
data.seurat.multiome2@meta.data <- data.seurat.multiome2@meta.data %>%
    as.data.frame() %>%
    rownames_to_column(var="barcode") %>%
    mutate(barcode = paste0(map_chr(barcode, ~ strsplit(.x, "-")[[1]][1]), "-ITBOGE013_YBP2_14_Multiome_2")) %>%
    column_to_rownames(var="barcode")

In [ ]:
data.seurat.multiome3 <- readRDS(paste0(io$outdir.processed, "/10X.multiome.processed._20250227_190214.annotated.rds"))

In [ ]:
data.seurat.multiome3 <- subset(data.seurat.multiome3, subset = condition %in% c("Control"))

In [ ]:
data.seurat.multiome3@meta.data <- data.seurat.multiome3@meta.data %>% 
    rownames_to_column(var="barcode") %>%
    mutate(barcode = paste0(map_chr(barcode, ~ strsplit(.x, "-")[[1]][1]), "-ITBOGE001_Fujii")) %>%
    column_to_rownames(var="barcode")

### Export to h5ad

In [ ]:
slim_seurat <- function(data.seurat.multiome){
    rna_counts <- GetAssayData(data.seurat.multiome[["RNA"]], layer = "counts")
    cell_metadata <- data.seurat.multiome@meta.data  
    return(CreateSeuratObject(counts = rna_counts, meta.data = cell_metadata))
}

In [ ]:
seurat_obj <- slim_seurat(data.seurat.multiome1)

In [ ]:
filename <- paste0(io$outdir.processed, "/10X.multiome.processed.controls.ITBOGE013_YBP2_14_Multiome_1.h5ad")

In [ ]:
seurat_to_anndata(seurat_obj, filename, assay="RNA", layer="counts")

In [ ]:
seurat_obj <- slim_seurat(data.seurat.multiome2)

In [ ]:
filename <- paste0(io$outdir.processed, "/10X.multiome.processed.controls.ITBOGE013_YBP2_14_Multiome_2.h5ad")

In [ ]:
seurat_to_anndata(seurat_obj, filename, assay="RNA", layer="counts")

In [ ]:
seurat_obj <- slim_seurat(data.seurat.multiome3)

In [ ]:
filename <- paste0(io$outdir.processed, "/10X.multiome.processed.controls.ITBOGE001_Fujii.h5ad")

In [ ]:
seurat_to_anndata(seurat_obj, filename, assay="RNA", layer="counts")